## 1: Import Libraries and Load Dataset

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
import joblib

# Load the preprocessed dataset
print("Loading dataset...")
df = pd.read_csv(r"C:\Users\91904\Desktop\ksfe\AI\CODE\review_rating_system\data\model_learning_data.csv")
df = df.dropna(subset=['Text', 'Score'])

# Define Features (X) and Target (y)
X = df['Text']
y = df['Score']

Loading dataset...


## 2: Split Data into Train and Test Sets

In [3]:
# Train-Test Split (80% training, 20% testing)
print("Splitting data into train and test sets...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Splitting data into train and test sets...


## 3: Vectorize Text Data (TF-IDF)

Generally split the data into training and testing sets before fitting TF-IDF, because fitting TF-IDF on the entire dataset can cause data leakage.

In [4]:
# Text Vectorization using TF-IDF
print("Vectorizing text data (TF-IDF)...")
vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

Vectorizing text data (TF-IDF)...


## 4: Train, Evaluate, and Compare Models with GridSearchCV
Apply hyperparameter tuning via GridSearchCV on Logistic Regression, Random Forest, and Linear SVC, then compare their performance on the test set.

In [5]:
# 1. Define hyperparameter grids for each model
lr_param_grid = {
    'C': [0.1, 1.0, 10.0],
    'max_iter': [1000]
}

rf_param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [20, None]
}

svc_param_grid = {
    'C': [0.1, 1.0, 10.0]
}

# 2. Wrap models inside GridSearchCV
models = {
    "Logistic Regression": GridSearchCV(LogisticRegression(random_state=42), lr_param_grid, cv=3, scoring='accuracy', n_jobs=-1),
    "Random Forest": GridSearchCV(RandomForestClassifier(random_state=42), rf_param_grid, cv=3, scoring='accuracy', n_jobs=-1),
    "SVM (Linear SVC)": GridSearchCV(LinearSVC(random_state=42), svc_param_grid, cv=3, scoring='accuracy', n_jobs=-1)
}

best_model_name = ""
best_accuracy = 0.0
best_model = None

# 3. Train, Evaluate, and Compare Side-by-Side
print("\n--- Model Comparison with GridSearchCV Results ---")
for name, grid_search in models.items():
    print(f"Running GridSearchCV for {name}...")
    grid_search.fit(X_train_vec, y_train)
    
    print(f"-> Best Parameters for {name}: {grid_search.best_params_}")
    
    y_pred = grid_search.predict(X_test_vec)
    acc = accuracy_score(y_test, y_pred)
    print(f"-> {name} Test Accuracy: {acc * 100:.2f}%\n")
    
    # Track the global winner
    if acc > best_accuracy:
        best_accuracy = acc
        best_model_name = name
        best_model = grid_search.best_estimator_

print(f"Overall Best Model: {best_model_name} with {best_accuracy * 100:.2f}% Accuracy!")


--- Model Comparison with GridSearchCV Results ---
Running GridSearchCV for Logistic Regression...
-> Best Parameters for Logistic Regression: {'C': 1.0, 'max_iter': 1000}
-> Logistic Regression Test Accuracy: 72.83%

Running GridSearchCV for Random Forest...
-> Best Parameters for Random Forest: {'max_depth': None, 'n_estimators': 50}
-> Random Forest Test Accuracy: 67.86%

Running GridSearchCV for SVM (Linear SVC)...
-> Best Parameters for SVM (Linear SVC): {'C': 0.1}
-> SVM (Linear SVC) Test Accuracy: 72.43%

Overall Best Model: Logistic Regression with 72.83% Accuracy!



## --- Model Comparison with GridSearchCV Results ---
Running GridSearchCV for Logistic Regression...
-> Best Parameters for Logistic Regression: {'C': 1.0, 'max_iter': 1000}
-> Logistic Regression Test Accuracy: 72.83%

Running GridSearchCV for Random Forest...
-> Best Parameters for Random Forest: {'max_depth': None, 'n_estimators': 50}
-> Random Forest Test Accuracy: 67.86%

Running GridSearchCV for SVM (Linear SVC)...
-> Best Parameters for SVM (Linear SVC): {'C': 0.1}
-> SVM (Linear SVC) Test Accuracy: 72.43%

## Overall Best Model: Logistic Regression with 72.83% Accuracy!

## 5: Automatically Save the Winning Model

In [7]:
# Save the Winner
model_path = r"C:\Users\91904\Desktop\ksfe\AI\CODE\review_rating_system\models\best_overall__model.pkl"
vectorizer_path = r"C:\Users\91904\Desktop\ksfe\AI\CODE\review_rating_system\models\tfidf_vectorizer__best.pkl"

joblib.dump(best_model, model_path)
joblib.dump(vectorizer, vectorizer_path)

print(f"\nWinning model successfully saved to: {model_path}")


Winning model successfully saved to: C:\Users\91904\Desktop\ksfe\AI\CODE\review_rating_system\models\best_overall__model.pkl
